# Data Preparation (Preparação dos dados)

## Biblioteca / Configuração

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos modulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação dos dados
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from configs.paths import *
from configs.function_basic import *

# Modelos / Machine Learning
from sklearn.ensemble import ExtraTreesClassifier

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
pd.set_option('display.width', None)

print('Ambiente Configurado')

## Parâmetros Globais

In [ ]:
# define a coluna alvo do modelo
TARGET = 'FPD'
# garante reprodutibilidade dos experimentos
RANDOM_STATE = 42
# percentual máximo de valores ausentes permitido para manter a variável
PERCENTUAL_MAX_FALTANTES = 20
# quantidade de features mais relevantes a considerar
TOP_N_FEATURES = 30

## Carregamento dos dados 

In [ ]:
# Carregar dados CORRIGIDOS conforme solicitado
train = pd.read_csv(PROCESSED_DIR / 'abt01_train.csv')
test = pd.read_csv(PROCESSED_DIR / 'abt01_test.csv')

print(f'📊 Treino: {train.shape}')
print(f'📊 Teste: {test.shape}')
print(f'\n🎯 Distribuição do Target (Treino):')
print(f"  Treino: {(train[TARGET].value_counts(normalize=True) * 100).round(2)}")

## Preparação dos Dados

In [ ]:
# Separar features e target
X_train = train.drop(TARGET, axis=1)
y_train = train[TARGET]

X_test = test.drop(TARGET, axis=1)
y_test = test[TARGET]

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')

# Garantir mesmas features em treino e teste
features_common = X_train.columns.intersection(X_test.columns)
X_train = X_train[features_common]
X_test = X_test[features_common]

print(f'\n✅ Features alinhadas: {len(features_common)}')

### Seleção de Variaveis

In [ ]:
# Aplicando no treino
train_fs = X_train

# define X e y para treino
X_train = train_fs
y_train = train[TARGET] 

In [ ]:
# Inicializa ExtraTrees para ranking de features
clf = ExtraTreesClassifier(n_estimators=500, max_depth=None, min_samples_leaf=10,
                           random_state=RANDOM_STATE, n_jobs=-1)  # n_jobs=-1 usa todos os núcleos

# Treina o modelo
clf.fit(X_train, y_train)  # ajusta o modelo aos dados

# Cria DataFrame com features e importâncias
feat_imp = pd.DataFrame({"feature": X_train.columns, "importance": clf.feature_importances_}) \
            .sort_values(by="importance", ascending=False)  # ordena por importância

# Seleciona top N features mais relevantes
selected_features_fs = feat_imp.head(TOP_N_FEATURES)["feature"].tolist()  # lista de features
selected_features_df = feat_imp.head(TOP_N_FEATURES)  # DataFrame com top features

# Exibe quantidade de features selecionadas
print(f"Número de features selecionadas: {len(selected_features_fs)}")

In [ ]:
# Formata as importâncias com 3 casas decimais
feat_imp_table = feat_imp.sort_values(by="importance", ascending=False)
feat_imp_table["importance"] = feat_imp_table["importance"].round(3)

# Plotar importância das features (top N)
top_feats = feat_imp_table.head(TOP_N_FEATURES)

plt.figure(figsize=(15, 50))
plt.barh(top_feats['feature'], top_feats['importance'])
plt.gca().invert_yaxis()  # feature mais importante no topo
plt.title('Top Features por Importância')
plt.xlabel('Importância')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
a

In [ ]:
# Manter no X_train apenas as colunas selecionadas
X_train = X_train[selected_features_fs].copy()

### Correlação

In [ ]:
# calcula correlação apenas dessas features numéricas
corr = X_train.select_dtypes(include='number').corr()

# máscara para mostrar só metade da matriz
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot_kws={"size": 9},
    cbar_kws={"shrink": 0.8}
)

plt.title("Heatmap de Correlação — Top Features")
plt.tight_layout()
plt.show()


In [ ]:
# Identificar pares de features com correlação absoluta acima do limiar

# calcula correlação apenas dessas features numéricas ignorando sinais
corr = X_train.select_dtypes(include='number').corr().abs()

# evita pares duplicados e remove a diagonal principal
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# percorre a matriz filtrada para encontrar pares altamente correlacionados
high_corr_pairs = [
    (row, col, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.9
]

# lista final de pares com alta correlação
high_corr_pairs

In [ ]:
# Remover a feature menos importante entre pares altamente correlacionados

# Conjunto para armazenar features marcadas para remoção (evita duplicidade)
to_drop = set()

# percorre cada par de features com alta correlação
for f1, f2, _ in high_corr_pairs:
    
    # recupera a importância da primeira feature
    imp_f1 = feat_imp_table.loc[
        feat_imp_table['feature'] == f1, 'importance'
    ].values[0]
    
    # recupera a importância da segunda feature
    imp_f2 = feat_imp_table.loc[
        feat_imp_table['feature'] == f2, 'importance'
    ].values[0]

    # compara importâncias e marca para remoção a menos relevante
    if imp_f1 < imp_f2:
        to_drop.add(f1)
    else:
        to_drop.add(f2)

# lista final de features sugeridas para remoção
list(to_drop)

In [ ]:
# Drop de colunas redundantes ou sem importância para feature selection
cols_to_drop = to_drop
X_train = X_train.drop(columns=cols_to_drop)

In [ ]:
X_train.shape

## Salvamento dos Dados Processados

In [ ]:
# Salvar lista de features (excluindo a target) para referência futura
selected_features = [c for c in X_train.columns if c != TARGET]
with open(ARTIFACT_DIR / 'selected_features.pkl', 'wb') as f:
    pickle.dump(selected_features, f)

print(f'   ✓ Lista de features salva em: {ARTIFACT_DIR / "selected_features.pkl"}')
print(f'\n✅ Todos os dados processados foram salvos')
